# 03 — MLP with a rollout loss

The same network, unrolled `k` steps inside the loss so the gradient sees the *composed* map.
`k = 1` is the control: it must reproduce topic 02, or the k axis means nothing.

Read the ranges, not the medians. The `k` axis moves less than the seed spread does.

In [1]:
import sys, json, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks'
                       else pathlib.Path.cwd()))
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt, torch
from l63 import ARTIFACTS, evaluate as E, plots as P
from l63.data import make_datasets
from run.report import clean, load_model, load_rows, summarise

gt = json.load(open(ARTIFACTS / 'ground_truth.json'))
S  = clean(summarise(load_rows()))
DATA = list(gt['datasets'])                      # 'ode', 'sde', 'sde015'

def num(v, w=6, p=2):
    """A ruler value, or an em dash where it is genuinely undefined."""
    if isinstance(v, dict):
        v = v.get('median')
    undefined = v is None or v != v          # None from clean(), bare NaN from the json
    return f'{"—":>{w}}' if undefined else f'{v:{w}.{p}f}'

def rng(v, p=2):
    if v is None or v.get('lo') is None or v['lo'] != v['lo']:
        return '—'
    return f"[{v['lo']:.{p}f}–{v['hi']:.{p}f}]"

print(len(S), 'model x dataset entries ·', len(DATA), 'datasets')

43 model x dataset entries · 3 datasets


## Numbers

In [2]:
for key in DATA:
    print(f"--- {key.upper()} ---")
    for k in (1, 4, 8, 16):
        s = S.get(f'03_rollout_k{k}_{key}')
        if s is None: continue
        print(f"  k={k:2d}  horizon {num(s['horizon'],4,0)} {rng(s['horizon'],0):>12s}   "
              f"chaos {num(s['chaos'])} {rng(s['chaos']):>14s}   "
              f"alive {num(s['alive'])} {rng(s['alive']):>14s}")
    s = S.get(f'02_mlp_{key}')
    if s: print(f"  02    horizon {num(s['horizon'],4,0)} {rng(s['horizon'],0):>12s}   "
                f"chaos {num(s['chaos'])} {rng(s['chaos']):>14s}   "
                f"alive {num(s['alive'])} {rng(s['alive']):>14s}   <- the k=1 control target")

--- ODE ---
  k= 1  horizon  336    [242–427]   chaos   0.99    [0.91–1.00]   alive   1.00    [1.00–1.00]
  k= 4  horizon  423    [367–482]   chaos   1.00    [0.99–1.01]   alive   1.00    [1.00–1.00]
  k= 8  horizon  362    [269–482]   chaos   0.99    [0.92–1.02]   alive   1.00    [1.00–1.00]
  k=16  horizon  333    [150–453]   chaos   1.00    [0.11–1.01]   alive   1.00    [0.03–1.00]
  02    horizon  336    [242–399]   chaos   1.00    [0.91–1.01]   alive   1.00    [1.00–1.00]   <- the k=1 control target
--- SDE ---
  k= 1  horizon   41      [40–41]   chaos   0.44   [-0.37–0.95]   alive   0.12    [0.00–1.00]
  k= 4  horizon   41      [37–44]   chaos   0.60   [-0.26–0.78]   alive   0.44    [0.00–0.88]
  k= 8  horizon   41      [29–41]   chaos  -0.16   [-0.44–0.88]   alive   0.00    [0.00–0.97]
  k=16  horizon   43      [41–49]   chaos  -0.27  [-0.45–-0.09]   alive   0.00    [0.00–0.00]
  02    horizon   41      [40–41]   chaos   0.42   [-0.37–0.92]   alive   0.19    [0.00–1.00]   <- the

## This model

In [3]:
KEY = 'ode'      # any of DATA
s = S['03_rollout_k8_' + KEY]
ref = gt['datasets'][KEY]
d = make_datasets(seed=0, kind=ref['kind'], b=ref['b'])
m, hist = load_model('03_rollout_k8_' + KEY + '_s' + str(s['rep_seed']))

print(f"{m.n_params:,} parameters, history {m.history}, figures show seed {s['rep_seed']}")
print()
print(f"{'ruler':16s}{'median':>8s}   range over seeds")
for k in ('horizon', 'spread', 'climate', 'climate_vs_truth', 'chaos', 'alive', 'lobe'):
    v = s[k]
    p = 0 if k == 'horizon' else 2
    print(f"  {k:14s}{num(v, 8, p)}   {rng(v, p)} over {v['n']} seeds")
print(f"\ntruth on this dataset:  climate {ref['truth_climate']:.2f}   "
      f"alive {ref['truth_alive']:.2f}   lobe {ref['truth_lobe']:.2f}   "
      f"ground truth usable {ref['floor_steps']} steps")
print(f"\nfirst steps, ||u_hat_n - u_n|| in Lorenz units:")
for i, e in enumerate(s['early'][:6], 1):
    print(f"  n={i}  {e:.3e}")

17,411 parameters, history 1, figures show seed 1

ruler             median   range over seeds
  horizon            362   [269–482] over 5 seeds
  spread               —   — over 0 seeds
  climate           4.27   [1.11–26.09] over 5 seeds
  climate_vs_truth    6.89   [1.78–42.05] over 5 seeds
  chaos             0.99   [0.92–1.02] over 5 seeds
  alive             1.00   [1.00–1.00] over 5 seeds
  lobe              0.65   [0.61–0.69] over 5 seeds

truth on this dataset:  climate 0.62   alive 1.00   lobe 0.63   ground truth usable 362 steps

first steps, ||u_hat_n - u_n|| in Lorenz units:
  n=1  3.741e-02
  n=2  7.557e-02
  n=3  1.102e-01
  n=4  1.412e-01
  n=5  1.732e-01
  n=6  2.059e-01


## Figures

Banked by `run/report.py`; regenerated here from the same checkpoint so the notebook and the deck cannot disagree.

In [4]:
P.loss_figure(hist, None, n_val_traj=8); plt.show()
P.arch_figure(m.spec(), "", m.n_params, None); plt.show()
long = d.raw(m.forecast(d.eval[:gt['n_long'], :m.history], gt['long_steps']))
P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()

/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50155/4260566466.py:1: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.loss_figure(hist, None, n_val_traj=8); plt.show()
/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50155/4260566466.py:2: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.arch_figure(m.spec(), "", m.n_params, None); plt.show()


/var/folders/pb/rz8q2sqx26z8m1lmk12bk5zm0000gn/T/ipykernel_50155/4260566466.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  P.lorenz_map_figure(long, d.raw(d.eval), None); plt.show()


## Findings

_Written after reading the numbers above._

- 
- 
- 